<a href="https://colab.research.google.com/github/harpuneet-k/Celebal-Assignments/blob/main/Assignment5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 2.1 MB/s eta 0:00:00


In [11]:
# Import necessary libraries
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import (
    OneHotEncoder, StandardScaler, PolynomialFeatures
)
from sklearn.impute import SimpleImputer
from category_encoders import TargetEncoder

# Function to load data from a CSV file
def load_data(path):
    # Reads the CSV file and returns a pandas DataFrame
    df = pd.read_csv(path)
    return df

# Custom transformer to select specific columns in a pipeline
class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        # No fitting needed; just return self
        return self

    def transform(self, X):
        # Return only the selected columns
        return X[self.columns]

# Pipeline for processing numeric features
class NumericPipeline:
    def __init__(self, num_cols):
        self.num_cols = num_cols
        # Define pipeline: select columns → impute → scale
        self.pipeline = Pipeline([
            ('select', ColumnSelector(num_cols)),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ])

    def fit(self, X, y=None):
        self.pipeline.fit(X)
        return self

    def transform(self, X):
        return self.pipeline.transform(X)

# Pipeline for processing categorical features
class CategoricalPipeline:
    def __init__(self, cat_cols, encoding='onehot'):
        self.cat_cols = cat_cols
        # Choose encoding method
        if encoding == 'onehot':
            self.encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
        elif encoding == 'target':
            self.encoder = TargetEncoder()

        # Define pipeline: select columns → impute → encode
        self.pipeline = Pipeline([
            ('select', ColumnSelector(cat_cols)),
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('encode', self.encoder),
        ])

    def fit(self, X, y=None):
        # TargetEncoder requires target values (y)
        if hasattr(self.encoder, 'fit_transform'):
            self.pipeline.fit(X, y)
        else:
            self.pipeline.fit(X)
        return self

    def transform(self, X):
        return self.pipeline.transform(X)

# Custom domain-based feature engineering
def feature_engineering(df):
    """
    Adds new features:
      - TotalSF: total square footage of the house
      - HouseAge: how old the house is when sold
      - Qual_x_Area: interaction term (Overall Quality × Living Area)
    Also log-transforms skewed features to reduce outliers/skew.
    """
    df = df.copy()  # Avoid modifying original

    # New features
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['Qual_x_Area'] = df['OverallQual'] * df['GrLivArea']

    # Log-transform skewed features
    skewed = ['LotArea', 'GrLivArea', 'TotalSF']
    for col in skewed:
        df[col] = np.log1p(df[col])  # log(1 + x) to handle 0 values

    return df

# Build the full preprocessing pipeline
def build_preprocessing_pipeline(numeric_cols, categorical_cols, use_target_encoding=False):
    # Numeric pipeline with polynomial feature expansion (degree 2)
    num_pipeline = Pipeline([
        ('select', ColumnSelector(numeric_cols)),
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),  # Adds interaction terms
    ])

    # Categorical pipeline using either One-Hot or Target encoding
    cat_pipeline = CategoricalPipeline(
        categorical_cols,
        encoding='target' if use_target_encoding else 'onehot'
    )

    # Combine both pipelines
    full_pipeline = FeatureUnion([
        ('num', num_pipeline),
        ('cat', cat_pipeline.pipeline),
    ])
    return full_pipeline

# Main Execution: Load, Engineer, Preprocess
if __name__ == '__main__':
    # Step 1: Load training data
    df = load_data('/content/train.csv')

    # Step 2: Apply feature engineering
    df = feature_engineering(df)

    # Step 3: Identify numeric and categorical columns
    num_cols = df.select_dtypes(include=[np.number]).drop(columns=['SalePrice']).columns.tolist()
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()

    # Step 4: Build and fit the preprocessing pipeline
    pipeline = build_preprocessing_pipeline(num_cols, cat_cols, use_target_encoding=True)

    # Step 5: Transform the training data
    X_preprocessed = pipeline.fit_transform(df, df['SalePrice'])

    # Final Output
    print("✅ Preprocessing complete. Shape of transformed data:", X_preprocessed.shape)


✅ Preprocessing complete. Shape of transformed data: (1460, 903)


In [12]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 1. Load and preprocess training data
df = load_data('/content/train.csv')
df = feature_engineering(df)

# Split into features and target
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

# Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify numeric and categorical columns
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# 2. Build and fit the pipeline on training data
pipeline = build_preprocessing_pipeline(numeric_cols, categorical_cols, use_target_encoding=True)
X_train_transformed = pipeline.fit_transform(X_train, y_train)
X_val_transformed = pipeline.transform(X_val)

# 3. Train a regression model
model = Ridge(alpha=1.0)
model.fit(X_train_transformed, y_train)

# 4. Predict and evaluate
y_pred = model.predict(X_val_transformed)

# 5. Print metrics
mae = mean_absolute_error(y_val, y_pred)
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, y_pred)

print(f"Validation Metrics:")
print(f"MAE  (Mean Absolute Error) : {mae:.2f}")
print(f"MSE  (Mean Squared Error)  : {mse:.2f}")
print(f"RMSE (Root MSE)            : {rmse:.2f}")
print(f"R² Score                   : {r2:.4f}")


Validation Metrics:
MAE  (Mean Absolute Error) : 25948.19
MSE  (Mean Squared Error)  : 1681478786.34
RMSE (Root MSE)            : 41005.84
R² Score                   : 0.7808
